# Market Differentiated Bars

Calculate a fixed-width fractional difference of the observed AAPL dollar-bar log price, selecting the minimum tested order that passes the ADF 5% critical value.

## Process the Data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_preprocessing.market_differentiated_bars import (
    fractional_difference_fixed_width,
    plot_min_ffd,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
feature_dir = PROJECT_ROOT / "data/research_data/market/features"
period = "2025-01-01_2025-12-31"
dollar_bar_path = feature_dir / f"aapl_dollar_bar_{period}.parquet"
fractional_path = feature_dir / f"aapl_dollar_bar_fractional_{period}.parquet"

if not fractional_path.is_file():
    bars = pd.read_parquet(dollar_bar_path, columns=["end", "close"])
    log_close = np.log(bars[["close"]].astype(float)).rename(
        columns={"close": "log_close"}
    )
    weight_cutoff = 0.01
    diagnostics = plot_min_ffd(
        log_close,
        weight_cutoff=weight_cutoff,
        differencing_orders=np.linspace(0.0, 1.0, 11),
    )
    stationary_orders = diagnostics.index[
        diagnostics["adf_statistic"] < diagnostics["critical_value_5pct"]
    ]
    if stationary_orders.empty:
        raise ValueError("No tested differencing order passed the ADF 5% critical value.")
    selected_order = float(stationary_orders.min())
    differentiated = fractional_difference_fixed_width(
        log_close,
        differencing_order=selected_order,
        weight_cutoff=weight_cutoff,
    )
    fractional_bars = bars.loc[differentiated.index, ["end"]].copy()
    fractional_bars["fractionally_differenced_log_close"] = differentiated["log_close"]
    fractional_bars.to_parquet(fractional_path, index=False)

fractional_bars = pd.read_parquet(fractional_path)
fractional_path

## Take a Quick Look at the Data Structure

In [ ]:
fractional_bars.head()

In [ ]:
fractional_bars.info()

In [ ]:
fractional_bars.dtypes.value_counts()

In [ ]:
fractional_bars[["fractionally_differenced_log_close"]].describe()

In [ ]:
fractional_bars[["fractionally_differenced_log_close"]].hist(
    bins=50, figsize=(6, 4)
)
plt.tight_layout()
plt.show()